# Data exploration

In [1]:
from pathlib import Path

data_dir = Path("../data/fashion_mnist")

for file_path in sorted(data_dir.iterdir()):
    print(
        f"File: {file_path.name} | "
        f"Size: {file_path.stat().st_size / (1024 * 1024):.2f} MB"
    )

File: t10k-images-idx3-ubyte.gz | Size: 4.22 MB
File: t10k-labels-idx1-ubyte.gz | Size: 0.00 MB
File: train-images-idx3-ubyte.gz | Size: 25.20 MB
File: train-labels-idx1-ubyte.gz | Size: 0.03 MB


In [1]:
import gzip
from pathlib import Path

data_dir = Path("../data/fashion_mnist")

image_file = data_dir / "train-images-idx3-ubyte.gz"
label_file = data_dir / "train-labels-idx1-ubyte.gz"

with gzip.open(image_file, "rb") as file:
    image_header = file.read(16)

with gzip.open(label_file, "rb") as file:
    label_header = file.read(8)

print("Image header bytes:", image_header)
print("Image header length:", len(image_header))

print("\nLabel header bytes:", label_header)
print("Label header length:", len(label_header))

Image header bytes: b'\x00\x00\x08\x03\x00\x00\xea`\x00\x00\x00\x1c\x00\x00\x00\x1c'
Image header length: 16

Label header bytes: b'\x00\x00\x08\x01\x00\x00\xea`'
Label header length: 8


In [2]:
import gzip
import struct
from pathlib import Path

data_dir = Path("../data/fashion_mnist")

image_file = data_dir / "train-images-idx3-ubyte.gz"
label_file = data_dir / "train-labels-idx1-ubyte.gz"

# Decode the image header
with gzip.open(image_file, "rb") as file:
    image_header = file.read(16)

magic_number, num_images, rows, columns = struct.unpack(
    ">IIII", image_header
)

# Decode the label header
with gzip.open(label_file, "rb") as file:
    label_header = file.read(8)

label_magic_number, num_labels = struct.unpack(
    ">II", label_header
)

print("IMAGE FILE")
print("Magic number:", magic_number)
print("Number of images:", num_images)
print("Rows per image:", rows)
print("Columns per image:", columns)

print("\nLABEL FILE")
print("Magic number:", label_magic_number)
print("Number of labels:", num_labels)

print("\nDo image and label counts match?", num_images == num_labels)

IMAGE FILE
Magic number: 2051
Number of images: 60000
Rows per image: 28
Columns per image: 28

LABEL FILE
Magic number: 2049
Number of labels: 60000

Do image and label counts match? True


In [ ]:
#Load and Decode Fashion-MNIST Files
import gzip
import struct
from pathlib import Path

import numpy as np

data_dir = Path("../data/fashion_mnist")


def load_images(file_path):
    """Load images from a compressed Fashion-MNIST IDX file."""

    with gzip.open(file_path, "rb") as file:
        # Read the 16-byte image header.
        header = file.read(16)

        magic_number, num_images, rows, columns = struct.unpack(
            ">IIII", header
        )

        # Validate the file format.
        if magic_number != 2051:
            raise ValueError(
                f"Expected image magic number 2051, got {magic_number}"
            )

        # Read all remaining bytes containing the pixel values.
        image_data = file.read()

    # Convert bytes into an array of unsigned 8-bit integers.
    images = np.frombuffer(image_data, dtype=np.uint8)

    # Arrange pixels into individual images.
    images = images.reshape(num_images, rows, columns)

    return images


def load_labels(file_path):
    """Load labels from a compressed Fashion-MNIST IDX file."""

    with gzip.open(file_path, "rb") as file:
        # Read the 8-byte label header.
        header = file.read(8)

        magic_number, num_labels = struct.unpack(">II", header)

        # Validate the file format.
        if magic_number != 2049:
            raise ValueError(
                f"Expected label magic number 2049, got {magic_number}"
            )

        # Read the remaining bytes containing the labels.
        label_data = file.read()

    labels = np.frombuffer(label_data, dtype=np.uint8)

    # Verify the number of labels matches the header.
    if len(labels) != num_labels:
        raise ValueError(
            f"Expected {num_labels} labels, found {len(labels)}"
        )

    return labels